# Counting Across Time, and What the Corpus Is Made Of

**Computational Text Analysis Pack · Module 22 — Distant Reading and Its Critics**

*Technique:* Culturomics; corpus-composition bias; the Ngram Viewer critiques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/11-distant-reading-and-its-critics/22a_culturomics_and_composition.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Michel et al.'s 2011 *Science* paper launched "culturomics" on a corpus of
**5,195,769 digitised books** — roughly 4% of all books ever published, over 500
billion words. The Google Books Ngram Viewer made the method famous: plot a
word's frequency over time and read the curve as cultural history.

The critiques arrived quickly, and the most damaging is Pechenick, Danforth and
Dodds (2015): **the corpus composition changes over time**, so a rising curve may
be telling you what Google scanned rather than what people wrote. Scientific
literature is heavily over-represented in the twentieth century, so words common
in scientific prose rise for reasons that have nothing to do with culture.

We cannot fix that. We can learn to detect it, which is what this notebook is
for.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# Two snapshots, about a century apart. Neither is a continuous series, and
# saying so is the first methodological move.
old = df_colonial
new = df

def rates(frame, col="text"):
    c = Counter()
    for t in frame[col]:
        c.update(tokenise(t))
    n = sum(c.values())
    return c, n

c_old, n_old = rates(old)
c_new, n_new = rates(new[new["language"] == "English"])
print(f"pre-1929 prose about Africa : {n_old:,} tokens, {len(c_old):,} types")
print(f"2020s English African news  : {n_new:,} tokens, {len(c_new):,} types")
print()
print("These are not two points on one timeline. They are two different genres,")
print("collected differently, a century apart. Every comparison below is")
print("confounded by that, and the notebook's job is to keep saying so.")

## The culturomics move

Take a word, compute its rate per million in each snapshot, and look at the
change. This is the entire method, and it is genuinely informative — as long as
you stay aware of what else changed.

In [ ]:
def per_million(word):
    return (c_old[word] / n_old * 1e6, c_new[word] / n_new * 1e6)

WORDS = ["native", "tribe", "savage", "empire", "colony", "development",
         "government", "trade", "war", "school", "woman", "technology"]
rows = []
for w in WORDS:
    a, b = per_million(w)
    rows.append({"word": w, "pre_1929": a, "modern": b,
                 "change": (b - a) / a * 100 if a else float("inf")})
shift = pd.DataFrame(rows).sort_values("change")
print(shift.to_string(index=False, float_format=lambda v: f"{v:.1f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(shift))
ax.barh(y - 0.2, shift["pre_1929"], height=.4, label="pre-1929 prose")
ax.barh(y + 0.2, shift["modern"], height=.4, label="2020s news")
ax.set_yticks(y)
ax.set_yticklabels(shift["word"])
ax.set_xscale("symlog")
ax.set_xlabel("occurrences per million words (log scale)")
ax.legend()
ax.set_title("Two snapshots, one century apart")
plt.tight_layout()

## The Pechenick problem, on our own data

*savage* and *tribe* collapse; *technology* appears from nowhere. It is tempting
to read that as the decline of colonial vocabulary and the arrival of the
modern world — and some of it is exactly that.

But our two corpora differ in **genre**, not just date. One is book-length
narrative prose; the other is news reporting. Before making any cultural claim,
measure how much of the difference genre alone can explain.

In [ ]:
# A genre control: news vs news, same period, different topic categories.
pol = new[(new["language"] == "English") & (new["category"] == "politics")]
hea = new[(new["language"] == "English") & (new["category"] == "health")]
c_pol, n_pol = rates(pol)
c_hea, n_hea = rates(hea)

print(f"{'word':<14}{'century gap':>14}{'genre gap (same year)':>24}")
print("-" * 54)
for w in ["government", "development", "school", "woman", "trade", "war"]:
    a, b = per_million(w)
    century = abs(b - a)
    p = c_pol[w] / n_pol * 1e6 if n_pol else 0
    h = c_hea[w] / n_hea * 1e6 if n_hea else 0
    print(f"{w:<14}{century:>14.1f}{abs(p - h):>24.1f}")
print()
print("Where the two columns are comparable, the 'century' difference is within")
print("the range that topic alone produces inside a single year. That word will")
print("not support a claim about historical change from this data.")

In [ ]:
# The other half of Pechenick: what is the corpus actually made of?
print("Composition of the pre-1929 corpus, by author:\n")
comp = old.groupby("author").size().sort_values(ascending=False)
top = comp.head(8)
for a, n in top.items():
    print(f"   {str(a)[:44]:<46}{n:>5} passages  ({n/len(old):>5.1%})")
print(f"\n   top 8 authors account for {top.sum()/len(old):.1%} of the corpus")
print()
print("A frequency curve over this corpus is substantially a curve about those")
print("authors. Michel et al. had the same problem at a vastly larger scale, and")
print("scale does not solve it — it hides it.")

### 🔧 Your turn 1

Find a word whose century-gap is large AND whose genre-gap is small. How confident are you in it as a historical claim?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The strongest candidates are usually words tied to institutions that genuinely
did not exist in one period — *technology*, *internet*, *airport* — and for those
the claim is safe but also trivial, which is precisely Nan Da's charge from
notebook 11c: what is robust is obvious. The interesting words, the ones
carrying an argument about colonial discourse, are almost always the ones where
genre and period are hopelessly entangled in a corpus like this. The honest
output is a smaller, better-hedged claim, and a note about what corpus would be
needed to do better.

</details>